In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("df_100.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 14 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   номер_заказа                      104 non-null    str    
 1   дата_заказа                       104 non-null    str    
 2   дата_рождения                     104 non-null    str    
 3   пол                               104 non-null    str    
 4   имя_исследования                  104 non-null    str    
 5   имя_параметра                     104 non-null    str    
 6   анализатор                        104 non-null    str    
 7   результат                         104 non-null    str    
 8   нормальное_минимальное_значение   104 non-null    float64
 9   нормальное_максимальное_значение  104 non-null    float64
 10  референсное_значение              104 non-null    str    
 11  комментарий                       0 non-null      float64
 12  статус_результата  

In [4]:
df

,номер_заказа,дата_заказа,дата_рождения,пол,имя_исследования,имя_параметра,анализатор,результат,нормальное_минимальное_значение,нормальное_максимальное_значение,референсное_значение,комментарий,статус_результата,флаг
0,092-114606,2026-07-27 08:53:49.722,1949-02-06 00:00:00.000,Жен,Общий анализ крови с формулой,Эритроциты,Mindray CAL8000,"5,13",3.5,5.2,"3,5-5,2",NaN,Normal,NaN
1,092-114606,2026-07-27 08:53:49.722,1949-02-06 00:00:00.000,Жен,Общий анализ крови с формулой,Лейкоциты,Mindray CAL8000,"9,33",4.0,10.0,4-10,NaN,Normal,NaN
2,092-114606,2026-07-27 08:53:49.722,1949-02-06 00:00:00.000,Жен,Общий анализ крови с формулой,"Нейтрофилы, абс.",Mindray CAL8000,"6,07",1.8,7.7,"1,8-7,7",NaN,Normal,NaN
3,092-114606,2026-07-27 08:53:49.722,1949-02-06 00:00:00.000,Жен,Общий анализ крови с формулой,"Лимфоциты, %",Mindray CAL8000,"25,6",19.0,37.0,19-37,NaN,Normal,NaN
4,092-114606,2026-07-27 08:53:49.722,1949-02-06 00:00:00.000,Жен,Общий анализ крови с формулой,"Моноциты, %",Mindray CAL8000,"6,6",3.0,12.0,3-12,NaN,Normal,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,194-055264,2026-07-06 08:17:27.137,1976-06-30 00:00:00.000,Жен,Общий анализ крови с формулой,IG - незрелые гранулоциты,Mindray CAL8000,"0,1",0.0,2.0,0-2,NaN,Normal,NaN
100,194-055264,2026-07-06 08:17:27.137,1976-06-30 00:00:00.000,Жен,Общий анализ крови с формулой,Тромбоциты,Mindray CAL8000,350,180.0,320.0,180-320,NaN,High,NaN
101,194-055264,2026-07-06 08:17:27.137,1976-06-30 00:00:00.000,Жен,Общий анализ крови с формулой,MPV (Средний объем тромбоцитов),Mindray CAL8000,"11,4",9.4,12.4,"9,4-12,4",NaN,Normal,NaN
102,194-055264,2026-07-06 08:17:27.137,1976-06-30 00:00:00.000,Жен,Общий анализ крови с формулой,PDW - ширина распределения тромбоцитов,Mindray CAL8000,16,10.0,20.0,10-20,NaN,Normal,NaN


In [5]:
import argparse
import json
import re
import sys
from datetime import datetime
 
import pandas as pd
import numpy as np


In [8]:

DATE_FORMAT_DAYFIRST = True  # дата_заказа / дата_рождения как DD.MM.YYYY
 
PLACEHOLDER_SOURCE_LIS = None
PLACEHOLDER_REAGENT_LOT = None
PLACEHOLDER_RUN_ID = None
PLACEHOLDER_PATIENT_PSEUDO_ID = None
PLACEHOLDER_UNIT = None
 
REF_RANGE_STR_RE = re.compile(
    r"(-?\d+(?:[.,]\d+)?)\s*[-–—]\s*(-?\d+(?:[.,]\d+)?)"
)
 
 
PARAM_NAME_MAP = {
    "Эритроциты": "RBC",
    "Лейкоциты": "WBC",
    "Нейтрофилы, абс.": "NEUT_abs",
    "Лимфоциты, %": "LYMPH_pct",
    "Моноциты, %": "MONO_pct",
    "Эозинофилы, %": "EO_pct",
    "Базофилы, %": "BASO_pct",
    "IG - незрелые гранулоциты": "IG",
    "Гемоглобин": "HGB",
    "Гематокрит": "HCT",
    "MCV (Средний объем эритроцитов в общем объеме пробы)": "MCV",
    "Лимфоциты, абс.": "LYMPH_abs",
    "Эозинофилы, абс.": "EO_abs",
    "Нейтрофилы, %": "NEUT_pct",
    "Моноциты, абс.": "MONO_abs",
    "Базофилы, абс.": "BASO_abs",
    "PCT (Тромбоцитокрит)": "PCT",
    "СОЭ (по Вестергрену)": "ESR",
    "MCH (Средний объем гемоглобина в эритроцитах)": "MCH",
    "MCHC (Средняя концентрация гемоглобина в эритроцитах)": "MCHC",
    "RDW-CV - ширина распределения эритроцитов (коэффициент отклонения)": "RDW_CV",
    "Тромбоциты": "PLT",
    "MPV (Средний объем тромбоцитов)": "MPV",
    "Палочкоядерные, %": "NEUT_band_pct",
    "Сегментоядерные, %": "NEUT_seg_pct",
    "PDW - ширина распределения тромбоцитов": "PDW",
}

def normalize_param_name(raw_name: str) -> str:
    """Обрезает пробелы по краям и схлопывает внутренние пробелы в один."""
    return re.sub(r"\s+", " ", str(raw_name).strip())
 
 
def map_param_name(raw_name: str) -> str:
    """
    Возвращает английскую аббревиатуру для русского имени параметра.
    Если название не найдено в словаре — не падаем, а возвращаем
    нормализованное исходное имя (чтобы новые/неизвестные параметры
    не терялись, а были заметны в выходном JSON и их можно было
    дописать в PARAM_NAME_MAP).
    """
    normalized = normalize_param_name(raw_name)
    return PARAM_NAME_MAP.get(normalized, normalized)
 

 
UNIT_MAP = {
    "RBC": "10^12/L",
    "WBC": "10^9/L",
    "NEUT_abs": "10^9/L",
    "LYMPH_pct": "%",
    "MONO_pct": "%",
    "EO_pct": "%",
    "BASO_pct": "%",
    "IG": "%",
    "HGB": "g/L",
    "HCT": "%",
    "MCV": "fL",
    "LYMPH_abs": "10^9/L",
    "EO_abs": "10^9/L",
    "NEUT_pct": "%",
    "MONO_abs": "10^9/L",
    "BASO_abs": "10^9/L",
    "PCT": "%",
    "ESR": "mm/h",
    "MCH": "pg",
    "MCHC": "g/dL",
    "RDW_CV": "%",
    "PLT": "10^9/L",
    "MPV": "fL",
    "NEUT_band_pct": "%",
    "NEUT_seg_pct": "%",
    "PDW": "%",
}
 
def get_unit(param_code: str):
    """Единица измерения по английскому коду параметра, либо None, если код неизвестен."""
    return UNIT_MAP.get(param_code)
 


# ----------------------------- Утилиты -----------------------------
 
def parse_date(value):
    """Парсит дату из строки CSV в datetime. Возвращает None при неудаче."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    value = str(value).strip()
    if not value:
        return None
    dt = pd.to_datetime(value, dayfirst=DATE_FORMAT_DAYFIRST, errors="coerce")
    if pd.isna(dt):
        return None
    return dt.to_pydatetime()
 
 
def to_iso_z(dt):
    """datetime -> ISO 8601 строка с суффиксом Z (без микросекунд)."""
    if dt is None:
        return None
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
 
 
def calc_age(birth_dt, order_dt):
    if birth_dt is None or order_dt is None:
        return None
    years = order_dt.year - birth_dt.year - (
        (order_dt.month, order_dt.day) < (birth_dt.month, birth_dt.day)
    )
    return years
 
 
def age_category(age_years):
    if age_years is None:
        return "unknown"
    if age_years < 18:
        return "child"
    if age_years < 60:
        return "adult"
    return "elderly"
 
 
def parse_ref_range(row):
    """
    Приоритет:
      1) числовые колонки нормальное_минимальное/максимальное_значение
      2) парсинг строки референсное_значение
      3) [null, null]
    """
    lo = row.get("нормальное_минимальное_значение")
    hi = row.get("нормальное_максимальное_значение")
 
    lo_ok = lo is not None and not (isinstance(lo, float) and np.isnan(lo))
    hi_ok = hi is not None and not (isinstance(hi, float) and np.isnan(hi))
    if lo_ok and hi_ok:
        return [float(lo), float(hi)]
 
    ref_str = row.get("референсное_значение")
    if isinstance(ref_str, str):
        m = REF_RANGE_STR_RE.search(ref_str.replace(",", "."))
        if m:
            try:
                return [float(m.group(1)), float(m.group(2))]
            except ValueError:
                pass
 
    return [None, None]
 
 
def to_number(value):
    """результат -> float, если возможно; иначе исходная строка (текстовый результат)."""
    if value is None:
        return None
    s = str(value).strip().replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return value
 
 
# ----------------------------- Основная сборка -----------------------------
 
def build_record(order_id, group: pd.DataFrame) -> dict:
    first = group.iloc[0]
 
    order_dt = parse_date(first.get("дата_заказа"))
    birth_dt = parse_date(first.get("дата_рождения"))
    age_years = calc_age(birth_dt, order_dt)
 
    indices = {}
    for _, row in group.iterrows():
        param_name = row.get("имя_параметра")
        if param_name is None or (isinstance(param_name, float) and np.isnan(param_name)):
            continue
        param_code = map_param_name(param_name)
        indices[param_code] = {
            "value": to_number(row.get("результат")),
            "unit": get_unit(param_code),
            "ref_range": parse_ref_range(row),
            # статус_результата -> flag, как договорились; колонку "флаг" не используем
            "flag": row.get("статус_результата") if row.get("статус_результата") not in (None, "") else None,
        }
 
    record = {
        "role": "user",
        "content": {
            "case_id": str(order_id),
            "created_at": to_iso_z(order_dt),
            "source_lis": PLACEHOLDER_SOURCE_LIS,
            "analyzer": {
                "model": first.get("анализатор"),
                "reagent_lot": PLACEHOLDER_REAGENT_LOT,
                "run_id": PLACEHOLDER_RUN_ID,
            },
            "patient_context": {
                "patient_pseudo_id": PLACEHOLDER_PATIENT_PSEUDO_ID,
                "age_years": age_years,
                "age_category": age_category(age_years),
                "sex": first.get("пол"),
                "clinical_diagnosis_codes": [],
                "clinical_diagnosis_text": "",
                "treatment_phase": "unknown",
                "ward_type": "unknown",
            },
            "current_result": {
                "collected_at": to_iso_z(order_dt),
                "indices": indices,
                "instrument_flags": [],
                "scattergram_abnormal": None,
                "morphology_findings": {
                    "source": "none",
                    "findings_text": "",
                    "immature_cell_pct": None,
                },
            },
            "historical_results": [],
            "delta_check": None,
            "rule_based_engine_output": None,
        },
    }
    return record
 
 
def run(input_csv: str, output_jsonl: str, encoding: str = "utf-8"):
    dtype_str_cols = [
        "номер_заказа", "дата_заказа", "дата_рождения", "пол",
        "имя_исследования", "имя_параметра", "анализатор", "результат",
        "референсное_значение", "комментарий", "статус_результата", "флаг",
    ]
    df = pd.read_csv(
        input_csv,
        dtype={c: str for c in dtype_str_cols},
        encoding=encoding
    )
 
    n_written = 0
    with open(output_jsonl, "w", encoding="utf-8") as f:
        for order_id, group in df.groupby("номер_заказа", sort=False):
            record = build_record(order_id, group)
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            n_written += 1
 
    print(f"Готово: {n_written} заказов -> {output_jsonl}", file=sys.stderr)
 
 
def main():
    
    run('df_100.csv', 'output_jsonl3.json', "utf-8")
 
 
if __name__ == "__main__":
    main()


/tmp/ipykernel_34790/3991868146.py:105: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(value, dayfirst=DATE_FORMAT_DAYFIRST, errors="coerce")
/tmp/ipykernel_34790/3991868146.py:105: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(value, dayfirst=DATE_FORMAT_DAYFIRST, errors="coerce")
/tmp/ipykernel_34790/3991868146.py:105: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(value, dayfirst=DATE_FORMAT_DAYFIRST, errors="coerce")
/tmp/ipykernel_34790/3991868146.py:105: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a forma